# Traitement des données et prise de connaissance des données

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import re

In [2]:
# Définition des chemins d'accès au données à traiter, et du chemin d'enregistrement pour les données traitées

path_raw_data = r"..\1. Données\01_raw"
path_data = path_raw_data + r"\welddb.data"
path_data_hearder = path_raw_data + r"\welddb.info"

path_data_processed = r"..\1. Données\02_intermediate"

## Formatage des données bruts sous forme .csv

In [3]:
# Lecture des infos sur la base de données 
with open (path_data_hearder, 'r') as f:
    text = f.read()

# Extraction des noms de features et des unités associées
lines = [l.strip() for l in text.splitlines() if l.strip]

names, units , role = [] , [], []

for i, line in enumerate(lines) : 
    if line.startswith("Column") : 
        description = lines[i+1]
        description = description.split(";")[0]

        if "/" in description :
            nom, unite = description.split('/', 1)

        else : 
            nom, unite = description , ""

        names.append(nom.strip())
        units.append(unite.strip())

# Affectation d'un rôle à chaque feature en fonction de son index dans la liste des features
for i, nom in enumerate(names) :
    if i in range(0,21) : 
        role.append("Molecular concentration")
    elif i in range(21, 28) :
        role.append("Physical process")
    elif i in range(28,30) : 
        role.append('Heat treatment')
    elif i in range(30,38) : 
        role.append("Test condition" if nom == "Charpy temperature" else "Quality indicator")
    elif i in range(38,43) : 
        role.append("Microstructure")
    elif i == 43 : 
        role.append("Id")

# Affectation du types de données à chaque feature 
list_of_string_features = ["AC or DC", "Electrode positive or negative", "Type of weld","Weld ID"]
types_of_feature = ["str" if nom in list_of_string_features else "float64" for nom in names]

# Création d'un dictionnaire de correspondance entre les noms de features et les unités
correspondance_between_names_units = pd.DataFrame({"Feature" : names, "Unit" : units, "Role" : role, "Types de variables" : types_of_feature})

In [4]:
# Lecture de la base de données et affectation des noms de features aux colonnes du dataframe
df = pd.read_csv(path_data, sep = r"\s+", header = None, names = names, na_values = "N")
df.shape

(1652, 44)

In [5]:
# Reformatage des features numériques qui sont mal formatées

def FormatWord(word) :
    """
    Convertit une valeur mal formatée en nombre (float).

    Les valeurs déjà numériques sont renvoyées telles quelles. Les formats
    spéciaux rencontrés dans la base de soudures sont traités ainsi :

    - "<0.01", "<5"  : limite de détection → on garde le seuil (0.01, 5)
    - "150-200"      : intervalle → on prend la moyenne (175)
    - "67tot33res", "54totndres", "158(Hv30)" : on garde le nombre
      en début de chaîne (67, 54, 158)

    Paramètres
    ----------
    word : str, float ou NaN
        Valeur d'une cellule du DataFrame.

    Retour
    ------
    float
        La valeur convertie, ou NaN si elle est manquante ou non reconnue.

    Exemples
    --------
    >>> FormatWord("0.07")        # 0.07
    >>> FormatWord("<0.01")       # 0.01
    >>> FormatWord("150-200")     # 175.0
    >>> FormatWord("67tot33res")  # 67.0
    """

    if pd.isna(word) : 
        return word

    try : 
        return float(word)
    except ValueError : 
        pass
    
    if word.startswith("<") :
        return word.split("<")[1]

    elif "-" in word and not word.startswith("-"):
        bas, haut = word.split("-")
        return (float(bas) + float(haut)) / 2

    elif re.match(r"^\d+(\.\d+)?", word):
        return float(re.match(r"^\d+(\.\d+)?", word).group())


def valeurs_non_numeriques(df, colonnes):
    """
Renvoie, pour chaque colonne, la liste des valeurs présentes
mais non convertibles en numérique.

Paramètres
----------
df : DataFrame
colonnes : liste des noms de colonnes à vérifier

Retour
------
dict {nom_colonne: [valeurs non convertibles]}
"""
    fautes = {}
    for nom in colonnes:
        presentes = df[nom].notna()
        non_convertibles = pd.to_numeric(df[nom], errors="coerce").isna()
        fautes[nom] = df.loc[presentes & non_convertibles, nom].unique().tolist()
    return fautes


# Récupération de la liste des features qui sont mal fomatés (en terme de data types)
list_wrong_formatted_features = correspondance_between_names_units[correspondance_between_names_units['Types de variables']
                                                                   != df.dtypes.values]['Feature'].tolist()

df[list_wrong_formatted_features] = df[list_wrong_formatted_features].map(FormatWord)
df[list_wrong_formatted_features] = df[list_wrong_formatted_features].apply(pd.to_numeric, errors="coerce")

dict_val = valeurs_non_numeriques(df,list_wrong_formatted_features)
for col in dict_val :
    assert dict_val[col] == []

In [6]:
# Uniformisation des unités (ppm en w%)
mask_unit1 = correspondance_between_names_units["Unit"] == "parts per million by weight"
list_of_features = correspondance_between_names_units.loc[mask_unit1, "Feature"]
df[list_of_features] = df[list_of_features] / 10000

In [7]:
# Enregistrement des données traitées
df.to_csv(path_data_processed + r"\welddb_processed.csv", index=False, sep = ";")
correspondance_between_names_units.to_csv(path_data_processed + r"\correspondance_between_names_units.csv", index=False, sep = ";")